# Configuration


In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split

In [44]:
# Load the dataset
accidents_df = pd.read_csv('../data/US_Accidents_March23_sampled_500k.csv')

# Data Preprocessing


In [45]:
# different feature groups we could investigate for clustering
WEATHER_COLUMNS = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", 
    "Wind_Speed(mph)", "Precipitation(in)", "Wind_Direction", "Weather_Condition"
]
TEMPORAL_COLUMNS = [
    "Start_Time", "End_Time", "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight",
    "Astronomical_Twilight"
]
GEOGRAPHIC_COLUMNS= [
    "Start_Lat", "Start_Lng", "End_Lat", "End_Lng", "City", "County", "State", 
    "Zipcode", "Country", "Timezone", "Airport_Code", "Street", "Distance(mi)"
]
INFRA_COLUMNS = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", 
    "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal"
]
# show what they like
display(accidents_df[WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS].info())
display(accidents_df[WEATHER_COLUMNS].head(2))
display(accidents_df[TEMPORAL_COLUMNS].head(2))
display(accidents_df[GEOGRAPHIC_COLUMNS].head(2))
display(accidents_df[INFRA_COLUMNS].head(2))

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 40 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Temperature(F)         489534 non-null  float64
 1   Wind_Chill(F)          370983 non-null  float64
 2   Humidity(%)            488870 non-null  float64
 3   Pressure(in)           491072 non-null  float64
 4   Visibility(mi)         488709 non-null  float64
 5   Wind_Speed(mph)        463013 non-null  float64
 6   Precipitation(in)      357384 non-null  float64
 7   Wind_Direction         488803 non-null  str    
 8   Weather_Condition      488899 non-null  str    
 9   Start_Time             500000 non-null  str    
 10  End_Time               500000 non-null  str    
 11  Sunrise_Sunset         498517 non-null  str    
 12  Civil_Twilight         498517 non-null  str    
 13  Nautical_Twilight      498517 non-null  str    
 14  Astronomical_Twilight  498517 non-null  str    

None

,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Wind_Direction,Weather_Condition
0,77.0,77.0,62.0,29.92,10.0,5.0,0.0,NW,Fair
1,45.0,43.0,48.0,29.91,10.0,5.0,0.0,W,Fair


,Start_Time,End_Time,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,2019-06-12 10:10:56,2019-06-12 10:55:58,Day,Day,Day,Day
1,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,Night,Night,Night,Night


,Start_Lat,Start_Lng,End_Lat,End_Lng,City,County,State,Zipcode,Country,Timezone,Airport_Code,Street,Distance(mi)
0,30.641211,-91.153481,NaN,NaN,Zachary,East Baton Rouge,LA,70791-4610,US,US/Central,KBTR,Highway 19,0.000
1,38.990562,-77.399070,38.990037,-77.398282,Sterling,Loudoun,VA,20164-2813,US,US/Eastern,KIAD,Forest Ridge Dr,0.056


,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal
0,False,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,False,False


In [46]:
# Exclude columns that are not realistic to impute but has substantial data loss
if "Wind_Chill(F)" in WEATHER_COLUMNS:
    WEATHER_COLUMNS.remove("Wind_Chill(F)")
for col in ["End_Lat", "End_Lng", "Zipcode", "Airport_Code"]:
    if col in GEOGRAPHIC_COLUMNS:
        GEOGRAPHIC_COLUMNS.remove(col)
display(accidents_df[WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS].info())

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 35 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Temperature(F)         489534 non-null  float64
 1   Humidity(%)            488870 non-null  float64
 2   Pressure(in)           491072 non-null  float64
 3   Visibility(mi)         488709 non-null  float64
 4   Wind_Speed(mph)        463013 non-null  float64
 5   Precipitation(in)      357384 non-null  float64
 6   Wind_Direction         488803 non-null  str    
 7   Weather_Condition      488899 non-null  str    
 8   Start_Time             500000 non-null  str    
 9   End_Time               500000 non-null  str    
 10  Sunrise_Sunset         498517 non-null  str    
 11  Civil_Twilight         498517 non-null  str    
 12  Nautical_Twilight      498517 non-null  str    
 13  Astronomical_Twilight  498517 non-null  str    
 14  Start_Lat              500000 non-null  float64

None

In [47]:
# Drop missing values that are not realistic to impute
preprocessing_df = accidents_df[
    WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS
].dropna(subset=["Sunrise_Sunset", "Timezone", "Visibility(mi)"])

# Convert Start_Time and End_Time to datetime
preprocessing_df["Start_Time"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed")
preprocessing_df["End_Time"] = pd.to_datetime(preprocessing_df["End_Time"], format="mixed")

# State-wise imputation on numeric weather features
# First do date-wise, then falls back to month-wise
preprocessing_df["Date"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed").dt.date
preprocessing_df["Month"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed").dt.month
IMPUTED_FEATURES = [
    "Precipitation(in)", "Humidity(%)", "Pressure(in)",
    "Wind_Speed(mph)", "Visibility(mi)", "Temperature(F)"
]
for col in IMPUTED_FEATURES:
    preprocessing_df[col] = (
        preprocessing_df[col]
        .fillna(preprocessing_df.groupby(["State", "Date"])[col].transform("median"))
        .fillna(preprocessing_df.groupby(["State", "Month"])[col].transform("median"))
    )

# drop the remaining NA values
preprocessing_df = preprocessing_df.dropna()
preprocessing_df.info()


<class 'pandas.DataFrame'>
Index: 483131 entries, 0 to 499999
Data columns (total 37 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Temperature(F)         483131 non-null  float64       
 1   Humidity(%)            483131 non-null  float64       
 2   Pressure(in)           483131 non-null  float64       
 3   Visibility(mi)         483131 non-null  float64       
 4   Wind_Speed(mph)        483131 non-null  float64       
 5   Precipitation(in)      483131 non-null  float64       
 6   Wind_Direction         483131 non-null  str           
 7   Weather_Condition      483131 non-null  str           
 8   Start_Time             483131 non-null  datetime64[ns]
 9   End_Time               483131 non-null  datetime64[ns]
 10  Sunrise_Sunset         483131 non-null  str           
 11  Civil_Twilight         483131 non-null  str           
 12  Nautical_Twilight      483131 non-null  str           
 13  

# Feature Engineering

### Questions we are trying to address: What are the characteristics of the accidents that occur under unusual conditions? How are they associated with higher severity?


In [48]:
# Weather Features
features_df = pd.DataFrame()
WEATHER_FEATURES = [
    "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)"
]

weather = preprocessing_df["Weather_Condition"].fillna("").str.lower()
features_df['has_rain'] = weather.str.contains(r"\brain\b|drizzle|drizziling|rainy",case=False,na=False).astype(int)
features_df['has_clouds'] = weather.str.contains(r"\bcloudy\b|cloud|clouds",case=False,na=False).astype(int)
features_df['has_snow'] = weather.str.contains("snow|snowy",case=False,na=False).astype(int)
features_df['is_sunny'] = weather.str.contains("clear",case=False,na=False).astype(int)
features_df['has_fog'] = weather.str.contains("fog|foggy",case=False,na=False).astype(int)

# add them to the features dataframe
features_df[WEATHER_FEATURES] = preprocessing_df[WEATHER_FEATURES]
for col in WEATHER_FEATURES:
    display(features_df[col].describe())

display(features_df['has_rain'].value_counts())
display(features_df['has_clouds'].value_counts())
display(features_df['has_snow'].value_counts())
display(features_df['is_sunny'].value_counts())
display(features_df['has_fog'].value_counts())


count    483131.00000
mean         61.66877
std          18.96021
min         -35.00000
25%          49.00000
50%          64.00000
75%          76.00000
max         207.00000
Name: Temperature(F), dtype: float64

count    483131.000000
mean         64.881071
std          22.794354
min           1.000000
25%          48.000000
50%          67.000000
75%          84.000000
max         100.000000
Name: Humidity(%), dtype: float64

count    483131.00000
mean         29.53935
std           1.00276
min           0.12000
25%          29.37000
50%          29.86000
75%          30.03000
max          38.44000
Name: Pressure(in), dtype: float64

count    483131.000000
mean          9.091365
std           2.711775
min           0.000000
25%          10.000000
50%          10.000000
75%          10.000000
max         130.000000
Name: Visibility(mi), dtype: float64

count    483131.000000
mean          7.639911
std           5.317509
min           0.000000
25%           4.600000
50%           7.000000
75%          10.000000
max         822.800000
Name: Wind_Speed(mph), dtype: float64

count    483131.000000
mean          0.009377
std           0.144435
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          10.090000
Name: Precipitation(in), dtype: float64

has_rain
0    448621
1     34510
Name: count, dtype: int64

has_clouds
0    305379
1    177752
Name: count, dtype: int64

has_snow
0    473061
1     10070
Name: count, dtype: int64

is_sunny
0    430925
1     52206
Name: count, dtype: int64

has_fog
0    476137
1      6994
Name: count, dtype: int64

In [49]:
# Temporal features

# times

# cyclical encoding for season features
SEASON_MAP = {
    12: 0, 1: 0, 2: 0,    # Winter
    3: 1, 4: 1, 5: 1,    # Spring
    6: 2, 7: 2, 8: 2,    # Summer
    9: 3, 10: 3, 11: 3  # Fall
}
preprocessing_df["Season"] = preprocessing_df["Start_Time"].dt.month.map(SEASON_MAP)
features_df["Season_Sin"] = np.sin(2 * np.pi * preprocessing_df["Season"] / 4)
features_df["Season_Cos"] = np.cos(2 * np.pi * preprocessing_df["Season"] / 4)

features_df["Year"] = preprocessing_df["Start_Time"].dt.year
features_df["IsRushHour"] = (
    preprocessing_df["Start_Time"].dt.hour.between(6, 9) | # early morning
    preprocessing_df["Start_Time"].dt.hour.between(16, 19) # late evening
).astype(int)
features_df["IsWeekend"] = (
    preprocessing_df["Start_Time"].dt.dayofweek >= 5 # Saturday or Sunday
).astype(int)
features_df["Sunrise_Sunset"] = preprocessing_df["Sunrise_Sunset"].map({"Night": 0, "Day": 1})

# print distribution of each newly created feature
display(features_df["IsRushHour"].value_counts())
display(features_df["IsWeekend"].value_counts())
display(features_df["Season_Sin"].value_counts())
display(features_df["Season_Cos"].value_counts())
display(features_df["Sunrise_Sunset"].value_counts())


/var/folders/4j/5hl0kjh1529gl0sgpv5h03_00000gn/T/ipykernel_3904/2606470640.py:22: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  preprocessing_df["Start_Time"].dt.dayofweek >= 5 # Saturday or Sunday


IsRushHour
0    243867
1    239264
Name: count, dtype: int64

IsWeekend
0    407127
1     76004
Name: count, dtype: int64

Season_Sin
 0.000000e+00    141038
-1.000000e+00    130682
 1.000000e+00    106755
 1.224647e-16    104656
Name: count, dtype: int64

Season_Cos
 1.000000e+00    141038
-1.836970e-16    130682
 6.123234e-17    106755
-1.000000e+00    104656
Name: count, dtype: int64

Sunrise_Sunset
1    334627
0    148504
Name: count, dtype: int64

In [50]:
# geographic features
GEOGRAPHIC_FEATURES = [
    "Start_Lat", "Start_Lng", "City", "County", "State",
    "Country", "Timezone", "Distance(mi)"
]

# Group anything that begins with letters, -, and numbers as a highway
features_df["Has_Highway"] = preprocessing_df["Street"].str.contains(r'^[A-Za-z]+-[0-9]+').astype(int)

# add remaining features
features_df[GEOGRAPHIC_FEATURES] = preprocessing_df[GEOGRAPHIC_FEATURES]

# print distribution of newly created features
features_df["Has_Highway"].value_counts()


Has_Highway
0    320045
1    163086
Name: count, dtype: int64

In [51]:
# infrastructure features
INFRA_FEATURES = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", 
    "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal"
]

features_df["Infra_Count"] = preprocessing_df[INFRA_FEATURES].sum(axis=1)
TRAFFIC_CONTROL_FEATURES = ["Traffic_Signal", "Stop", "Give_Way"]
features_df["Has_Traffic_control"] = preprocessing_df[TRAFFIC_CONTROL_FEATURES].any(axis=1).astype(int)
TRANSIT_FEATURES = ["Station", "Railway"]
features_df["Has_Transit"] = preprocessing_df[TRANSIT_FEATURES].any(axis=1).astype(int)
INTERSECTION_FEATURES = ["Crossing", "Junction", "Roundabout"]
features_df["Has_Intersection"] = preprocessing_df[INTERSECTION_FEATURES].any(axis=1).astype(int)


# print distribution
display(features_df["Infra_Count"].value_counts())
display(features_df["Has_Traffic_control"].value_counts())
display(features_df["Has_Transit"].value_counts())
display(features_df["Has_Intersection"].value_counts())

Infra_Count
0    338399
1     97129
2     38346
3      7803
4      1271
5       167
6        15
7         1
Name: count, dtype: int64

Has_Traffic_control
0    397091
1     86040
Name: count, dtype: int64

Has_Transit
0    467301
1     15830
Name: count, dtype: int64

Has_Intersection
0    393102
1     90029
Name: count, dtype: int64

# Feature Selection


In [52]:
# Remove redundant (highly correlated) features
# corr_matrix = features_df.corr().abs()
# print("Correlation Matrix before removing highly correlated features:")
# display(corr_matrix)
# selected_df = features_df.drop(columns=["Infra_Count"])
# print("Correlation Matrix after removing highly correlated features:")
# display(selected_df.corr().abs())

# Remove low-variance features
# selector = VarianceThreshold(threshold=0.01)
# X_selected = selector.fit_transform(selected_df)
# selector.get_support() # all of them are true, so no low-variance features to remove


# Model Training


In [55]:
#Split: 80% training, 20% remaining

X_train,X_remaining = train_test_split(features_df,test_size=0.20,random_state=67)

#Split the remaining 20% into 10% Validation and 10% Testing
X_val,X_test = train_test_split(X_remaining,test_size=0.50,random_state=67)

# Creating the model
model_prototype_1 = IsolationForest(random_state=67)

print(X_train.select_dtypes(exclude=["number", "bool"]).columns)

#Training the model only using the training set.
#model_prototype_1.fit(X_train)



Index(['City', 'County', 'State', 'Country', 'Timezone'], dtype='str')
